In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine

# 1) Cargar .env de forma robusta
env_path = find_dotenv(usecwd=True)  # busca .env desde la cwd hacia arriba
if not env_path:
    raise FileNotFoundError("No se encontró .env. Verifica ubicación/nombre.")
load_dotenv(env_path, override=True)

# 2) Leer variables y validar
required = ["MYSQL_HOST", "MYSQL_PORT", "MYSQL_DB", "MYSQL_USER", "MYSQL_PASSWORD"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(f"Faltan variables en .env: {missing}")

host = os.getenv("MYSQL_HOST")
port = int(os.getenv("MYSQL_PORT", "3306"))
db   = os.getenv("MYSQL_DB")
user = os.getenv("MYSQL_USER")
pwd  = os.getenv("MYSQL_PASSWORD")

print("Usando host:", host, "| db:", db, "| user:", user)  # no imprimas la clave

# 3) Crear engine y probar
engine = create_engine(
    f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}?charset=utf8mb4",
    pool_pre_ping=True
)

df = pd.read_sql("SELECT COUNT(*) AS filas FROM FactVentasDetalle", engine)
df


# Ejemplo: traer ventas por día (top 10)
q = """
SELECT f.Fecha, SUM(v.Total) AS VentasTotal
FROM FactVentasDetalle v
JOIN DimFecha f ON f.FechaID = v.FechaFacturaID
GROUP BY f.Fecha
ORDER BY f.Fecha DESC
LIMIT 10;
"""
preview = pd.read_sql(q, engine)
preview.head()




Usando host: 127.0.0.1 | db: dwh_dev | user: root


,Fecha,VentasTotal
0,2025-12-31,3264.33
1,2025-12-30,3320.83
2,2025-12-28,2535.05
3,2025-12-27,1431.01
4,2025-12-25,2122.64


In [3]:
#1) Helpers (reutilizables)
import pandas as pd
from sqlalchemy import text

def q(sql, params=None):
    """Ejecuta SQL y devuelve DataFrame."""
    return pd.read_sql(text(sql), engine, params=params or {})

def to_csv(df, path):
    df.to_csv(path, index=False)
    print(f"Guardado: {path}")

In [4]:
#KPI básicos (Ventas, Egresos, Margen) por día

sql_margen_dia = """
WITH v AS (
  SELECT v.FechaFacturaID, SUM(v.SinImpuesto) AS Ventas
  FROM FactVentasDetalle v
  GROUP BY v.FechaFacturaID
),
e AS (
  SELECT e.FechaFacturaID, SUM(e.SinImpuesto) AS Egresos
  FROM FactEgresosDetalle e
  GROUP BY e.FechaFacturaID
)
SELECT
  f.Fecha,
  COALESCE(v.Ventas,0)  AS Ventas,
  COALESCE(e.Egresos,0) AS Egresos,
  COALESCE(v.Ventas,0) - COALESCE(e.Egresos,0) AS Margen
FROM DimFecha f
LEFT JOIN v ON v.FechaFacturaID = f.FechaID
LEFT JOIN e ON e.FechaFacturaID = f.FechaID
WHERE f.Fecha BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY f.Fecha;
"""
df_margen_dia = q(sql_margen_dia)
df_margen_dia.head()


,Fecha,Ventas,Egresos,Margen
0,2025-01-01,2914.34,1248.84,1665.50
1,2025-01-02,0.00,0.00,0.00
2,2025-01-03,0.00,1911.12,-1911.12
3,2025-01-04,2396.92,0.00,2396.92
4,2025-01-05,976.87,0.00,976.87


In [5]:
#Top Productos por ventas y marge

sql_top_prod = """
WITH vent AS (
  SELECT ProductoID, SUM(SinImpuesto) AS Ventas
  FROM FactVentasDetalle
  GROUP BY ProductoID
),
eg AS (
  SELECT ProductoID, SUM(SinImpuesto) AS Egresos
  FROM FactEgresosDetalle
  GROUP BY ProductoID
)
SELECT
  p.ProductoID,
  p.NomProducto,
  COALESCE(vent.Ventas,0)  AS Ventas,
  COALESCE(eg.Egresos,0)   AS Egresos,
  COALESCE(vent.Ventas,0) - COALESCE(eg.Egresos,0) AS Margen
FROM DimProducto p
LEFT JOIN vent ON vent.ProductoID = p.ProductoID
LEFT JOIN eg   ON eg.ProductoID   = p.ProductoID
ORDER BY Margen DESC
LIMIT 20;
"""
df_top_prod = q(sql_top_prod)
df_top_prod


,ProductoID,NomProducto,Ventas,Egresos,Margen
0,3,Venta impresos,1054301.89,0.00,1054301.89
1,1,Venta comisionada,1051513.51,0.00,1051513.51
2,2,Venta validada,1002828.17,0.00,1002828.17
3,4,Venta alcance,993640.10,0.00,993640.10
4,5,Venta comision PD,983287.44,0.00,983287.44
5,9,Costos de venta,0.00,463542.83,-463542.83
6,6,Marketing,0.00,485988.00,-485988.00
7,11,RRHH,0.00,489023.57,-489023.57
8,7,TI,0.00,516553.15,-516553.15
9,8,Administración,0.00,527291.42,-527291.42


In [7]:
#Margen por Evento y Organizador (sin duplicar)

#Como un evento puede tener varios organizadores, si sumas directo por el bridge duplicas los importes.
#La forma correcta es prorratear por el número de organizadores del evento.

sql_margen_org = """
WITH cnt_org AS (
  SELECT EventoID, COUNT(*) AS n
  FROM eventosOrganizador_Detalle
  GROUP BY EventoID
),
v AS (
  SELECT EventoID, SUM(SinImpuesto) AS Ventas
  FROM FactVentasDetalle
  GROUP BY EventoID
),
e AS (
  SELECT EventoID, SUM(SinImpuesto) AS Egresos
  FROM FactEgresosDetalle
  GROUP BY EventoID
)
SELECT
  o.OrganizadorID,
  d.Nombre AS Organizador,
  SUM( COALESCE(v.Ventas,0)  / NULLIF(c.n,0) ) AS VentasAjustadas,
  SUM( COALESCE(e.Egresos,0) / NULLIF(c.n,0) ) AS EgresosAjustados,
  SUM( COALESCE(v.Ventas,0)  / NULLIF(c.n,0)
     - COALESCE(e.Egresos,0) / NULLIF(c.n,0) ) AS MargenAjustado
FROM eventosOrganizador_Detalle o
JOIN cnt_org c ON c.EventoID = o.EventoID
LEFT JOIN v     ON v.EventoID = o.EventoID
LEFT JOIN e     ON e.EventoID = o.EventoID
LEFT JOIN DimOrganizadores d ON d.OrganizadorID = o.OrganizadorID
GROUP BY o.OrganizadorID, d.Nombre
ORDER BY MargenAjustado DESC;
"""
df_margen_org = q(sql_margen_org)
df_margen_org.head()


,OrganizadorID,Organizador,VentasAjustadas,EgresosAjustados,MargenAjustado
0,3357,Carbajal de Alonso,9467.33,0.0,9467.33
1,3213,Duarte y Ureña SA,7017.08,0.0,7017.08
2,3516,Asociación Toro,6966.91,0.0,6966.91
3,2282,Air Nevárez SA,6941.39,0.0,6941.39
4,2652,Gálvez y Palomino y Flia.,6344.67,0.0,6344.67


In [8]:
#5) Precio promedio de ticket por evento

sql_precio_evento = """
SELECT
  ev.EventoID,
  ev.NombreEvento,
  SUM(v.Total)/NULLIF(SUM(v.CantidadProd),0) AS PrecioPromedio,
  SUM(v.CantidadProd) AS Unidades,
  SUM(v.SinImpuesto) AS Ventas
FROM FactVentasDetalle v
JOIN DimEventos ev ON ev.EventoID = v.EventoID
GROUP BY ev.EventoID, ev.NombreEvento
ORDER BY Ventas DESC
LIMIT 30;
"""
df_precio_evento = q(sql_precio_evento)
df_precio_evento.head()


,EventoID,NombreEvento,PrecioPromedio,Unidades,Ventas
0,3357,Voluptates earum unde ut.,171.868462,65.0,9467.33
1,2669,Voluptatem ipsa.,155.486212,66.0,8696.68
2,1941,Distinctio molestiae et et.,120.849855,69.0,7066.64
3,3213,Illum hic suscipit dolor.,142.761207,58.0,7017.08
4,3516,Eos porro ut.,124.559848,66.0,6966.91


In [9]:
#6) Guardar resultados
to_csv(df_margen_dia,   "out/margen_por_dia.csv")
to_csv(df_top_prod,     "out/top_productos.csv")
to_csv(df_margen_org,   "out/margen_por_organizador.csv")
to_csv(df_precio_evento,"out/precio_promedio_evento.csv")


Guardado: out/margen_por_dia.csv
Guardado: out/top_productos.csv
Guardado: out/margen_por_organizador.csv
Guardado: out/precio_promedio_evento.csv
